# 3a · Find the gestures

**What this notebook does:** It finds gestures automatically, from the speed of one hand. The idea is simple:

1. A gesture has at least one moment where the hand moves **fast** (a speed **peak**).
2. From each peak we go back in time to where the hand was last **at rest**: that's where the gesture starts.
   Then we go forward to where the hand is at rest again: that's where it ends.
3. Gestures with only a tiny pause between them are joined into one.

| You need | You get |
|---|---|
| `Upsampled/Test_Video_processed_data.csv` (from Notebook 2) | `Events/Test_Video_events_found.csv`: the data from Notebook 2 plus a `gesture_id` column that numbers each gesture (empty = no gesture at that moment). |

### What's a percentile?
The settings below use **percentiles** of the hand's speed in this video. The 70th percentile is the speed that
the hand is slower than 70% of the time. Because percentiles adapt to each video, the same settings work for calm
speakers and for lively ones.

## Step 1 · Load the tools

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.signal import find_peaks   # finds the peaks (hilltops) in a curve

## Step 2 · Settings

In [ ]:
VIDEO_NAME = "Test_Video"

# Whose speed to use. It must be one of the KEYPOINTS you chose in Notebook 2.
KEYPOINT = "right_wrist"

# A gesture must, at least once, be faster than this percentile of the video.
PEAK_PERCENTILE = 70

# The hand counts as "at rest" when it is slower than this percentile.
REST_PERCENTILE = 5

# Two gestures with a pause shorter than this (in milliseconds) between them are joined into one.
MIN_PAUSE_MS = 50

## Step 3 · Load the speed data from Notebook 2

In [ ]:
data = pd.read_csv(f"../Upsampled/{VIDEO_NAME}_processed_data.csv")
time = data["time_ms"].to_numpy()
speed = data[f"{KEYPOINT}_speed"].to_numpy()

peak_level = np.percentile(speed, PEAK_PERCENTILE)
rest_level = np.percentile(speed, REST_PERCENTILE)
print(f"Peak level: {peak_level:.3f}   Rest level: {rest_level:.3f}   (speed goes from 0 to 1)")

## Step 4 · Find the fast moments (peaks)

A **peak** is a moment where the speed is higher than just before and just after it, like a hilltop.
We keep only the peaks above the peak level.

In [ ]:
peaks, _ = find_peaks(speed, height=peak_level)
print(f"Found {len(peaks)} peaks above the peak level")

## Step 5 · Find where each gesture starts and ends

For each peak, we step back in time, one measurement at a time, until the hand is at rest. That's the start
(**onset**). Then we step forward from the peak until the hand is at rest again. That's the end (**offset**).

In [ ]:
gestures = []  # a list of (onset, offset) pairs, in milliseconds
for peak in peaks:
    start = peak
    while start > 0 and speed[start] > rest_level:               # step back until the hand rests
        start -= 1
    end = peak
    while end < len(speed) - 1 and speed[end] > rest_level:      # step forward until the hand rests
        end += 1
    gestures.append((time[start], time[end]))

## Step 6 · Tidy up

* A gesture with several peaks was found several times, once per peak. We keep one copy.
* Gestures separated by a pause shorter than `MIN_PAUSE_MS` are joined into one.

In [ ]:
gestures = sorted(set(gestures))  # set() removes the copies, sorted() puts them in time order

joined = []
for onset, offset in gestures:
    if joined and onset - joined[-1][1] < MIN_PAUSE_MS:   # too close to the previous gesture:
        joined[-1] = (joined[-1][0], max(joined[-1][1], offset))  # stretch that one instead
    else:
        joined.append((onset, offset))

gesture_table = pd.DataFrame(joined, columns=["onset", "offset"])
gesture_table.insert(0, "gesture_id", range(1, len(gesture_table) + 1))
gesture_table["duration"] = gesture_table["offset"] - gesture_table["onset"]
print(f"Found {len(gesture_table)} gestures")
gesture_table

## Step 7 · Label every moment with its gesture number

The table above lists the gestures. For the next notebooks we also write the gesture number next to every 10 ms
measurement that falls inside a gesture.

In [ ]:
data["gesture_id"] = np.nan  # nan = "no value": not part of a gesture
for gesture in gesture_table.itertuples():
    inside = (data["time_ms"] >= gesture.onset) & (data["time_ms"] <= gesture.offset)
    data.loc[inside, "gesture_id"] = gesture.gesture_id

## Step 8 · Check the result

The black line is the speed, the dots are the peaks, and each orange area is one gesture, with its number at the
top. The dashed line is the peak level, the dotted line the rest level. Use the slider below the plot to move
through the video.

**Not happy with the result?** Change the Settings and run the notebook again:
* Too many small gestures? Raise `PEAK_PERCENTILE`.
* Separate gestures got joined together? Lower `MIN_PAUSE_MS`.
* Gestures start too early or end too late? Raise `REST_PERCENTILE`.

In [ ]:
fig = go.Figure()
fig.add_scatter(x=time, y=speed, name="speed", line_color="black")
fig.add_scatter(x=time[peaks], y=speed[peaks], name="peaks", mode="markers",
                marker_color="royalblue")
for gesture in gesture_table.itertuples():
    fig.add_vrect(x0=gesture.onset, x1=gesture.offset, fillcolor="orange", opacity=0.3,
                  line_width=0, annotation_text=str(gesture.gesture_id),
                  annotation_position="top left")
fig.add_hline(y=peak_level, line_dash="dash", line_color="grey")
fig.add_hline(y=rest_level, line_dash="dot", line_color="grey")
fig.update_layout(title=f"Gestures found from the speed of the {KEYPOINT}", xaxis_title="Time (ms)",
                  yaxis_title="Speed (1 = fastest)", xaxis_rangeslider_visible=True)
fig.show()

## Step 9 · Save

In [ ]:
Path("../Events").mkdir(exist_ok=True)
output_path = f"../Events/{VIDEO_NAME}_events_found.csv"
data.to_csv(output_path, index=False)
print("Saved:", output_path)

## Done!

Continue with **Notebook 4**, which looks inside each of these gestures.